# 06 ? Integra??o de treino Chronos-2 com equation head

Este notebook conecta o modelo Chronos-2 ao nosso adaptador e ?s heads de skeleton embedding e escala de inova??o. O skeleton teacher fica congelado e fornece somente vetores de skeleton.

Registro: integra??o/piloto. N?o reporta m?tricas cient?ficas at? checkpoint, revis?o do Chronos e teacher Hugging Face estarem fixados e treino executado em hardware selecionado.

Chronos-2: `amazon/chronos-2` @ `29ec3766d36d6f73f0696f85560a422f50e8498c`. Teacher: `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` / `transformer_skeleton` @ `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. O teacher ? single-series e pode supervisionar seis canais somente com topologia self-only, relabel local da ?nica fonte para s?rie 0 e `prefix_values=0`. Topologias com liga??es entre canais seguem incompat?veis. O smoke com Chronos real usa teacher placeholder e valida apenas arquitetura/gradientes.

## Reproduction instructions

Use Python 3.11 from the repository root. Install the project dependencies with `python -m pip install -e ".[learning,dev,release]"`, then execute this notebook with:

`python -m nbconvert --to notebook --execute --inplace experiments/generator_model_evaluation/06_equation_head_training_integration.ipynb`

The Chronos-2 checkpoint used by this experiment is pinned to `amazon/chronos-2@29ec3766d36d6f73f0696f85560a422f50e8498c`; the skeleton teacher, where used, is pinned to `mateuswiteck/tsfmxai-canonicalization-encoder-decoder` variant `transformer_skeleton` at `2fcdcc88b41125bcfd92953c0bb8a668a603d167`. The notebook records its own seeds, settings, observed result, and current limitations.
The Chronos Python implementation is read from `experiments_local/chronos-forecasting`; recreate that ignored checkout at commit `10afa9ebe016e514f9d7dc1aa873f66af57e116b` and install it with `python -m pip install -e experiments_local/chronos-forecasting`.


In [1]:
from pathlib import Path
import sys
import torch
ROOT=Path.cwd()
while not (ROOT/'src/tsfmxai').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
if not (ROOT/'src/tsfmxai').exists(): raise RuntimeError('repository root not found')
sys.path.insert(0,str(ROOT/'src'))
from tsfmxai.datasets.chronos2_adapter import adapt_tsfmxai_batch
from tsfmxai.models.chronos2_equation import Chronos2EquationModel, equation_training_loss
from tsfmxai.pipelines.chronos2_equation_training import Chronos2TrainingSpec, load_equation_model, training_step
print(torch.__version__, 'CUDA:',torch.cuda.is_available())
from tsfmxai.pipelines.chronos2_equation_training import Chronos2TrainingSpec, training_step


2.12.0+cpu CUDA: False


## Configura??o fixada antes de treinar

Usar o mesmo Chronos-2 dos notebooks de forecasting, com revis?o imut?vel. `Chronos2TrainingSpec` exige ID e revis?o para impedir downloads flutuantes. N?o usar `Chronos2Pipeline.fit` para esta condi??o: ele n?o aceita nossa loss auxiliar customizada; o wrapper chama o modelo base e a loss quant?lica nativa, depois soma as duas perdas auxiliares.

In [2]:
MODEL_ID='amazon/chronos-2'
MODEL_REVISION='29ec3766d36d6f73f0696f85560a422f50e8498c'
TEACHER_REPO_ID='mateuswiteck/tsfmxai-canonicalization-encoder-decoder'
TEACHER_VARIANT='transformer_skeleton'
TEACHER_REVISION='2fcdcc88b41125bcfd92953c0bb8a668a603d167'
TEACHER_DIMENSION=96
OUTPUT_PATCH_SIZE=16
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
DEPENDENCY_PROBS={'independent':1.0}
TRAIN_CROSS_CHANNEL=False
# For each self-only equation, map its sole channel to local teacher series 0 and zero numeric values.
spec=Chronos2TrainingSpec(MODEL_ID,MODEL_REVISION,TEACHER_DIMENSION,OUTPUT_PATCH_SIZE,
                          embedding_weight=1.0,noise_weight=1.0,freeze_forecaster=False)
assert not TRAIN_CROSS_CHANNEL
print('pinned Chronos/teacher; six-channel self-only condition; CUDA=',torch.cuda.is_available())

pinned Chronos/teacher; six-channel self-only condition; CUDA= False


In [3]:
# Real Chronos-2 single-pass forward/backward smoke. Teacher vectors are placeholders, not scientific labels.
import torch, time
chronos_src=ROOT/'experiments_local/chronos-forecasting/src'
sys.path.insert(0,str(chronos_src))
from chronos.chronos2.pipeline import Chronos2Pipeline
torch.manual_seed(20261130)
pipeline=Chronos2Pipeline.from_pretrained(MODEL_ID,revision=MODEL_REVISION,device_map=DEVICE)
wrapped=Chronos2EquationModel(pipeline.model,skeleton_dimension=TEACHER_DIMENSION).to(DEVICE).train()
rows=12; context=torch.randn(rows,32,device=DEVICE); future=torch.randn(rows,16,device=DEVICE)
smoke_batch={'context':context,'future_target':future,'future_covariates':torch.full_like(future,float('nan')),
 'group_ids':torch.arange(2,device=DEVICE).repeat_interleave(6),'num_output_patches':1,
 'teacher_embedding':torch.randn(rows,TEACHER_DIMENSION,device=DEVICE),
 'noise_std':torch.rand(rows,device=DEVICE)*0.1+0.01}
t0=time.perf_counter(); smoke_output=wrapped(smoke_batch)
smoke_loss,smoke_metrics=equation_training_loss(smoke_output,smoke_batch)
smoke_loss.backward()
assert smoke_output['skeleton_embedding'].shape==(rows,TEACHER_DIMENSION)
assert smoke_output['predicted_log_noise_std'].shape==(rows,)
assert all(p.grad is not None for p in wrapped.skeleton_head.parameters())
assert all(p.grad is not None for p in wrapped.noise_head.parameters())
print({'device':DEVICE,'loss':float(smoke_loss.detach()),'forecast_loss':float(smoke_metrics['forecast']),
 'skeleton_shape':tuple(smoke_output['skeleton_embedding'].shape),
 'noise_shape':tuple(smoke_output['predicted_log_noise_std'].shape),
 'encoder_passes':1,'seconds':round(time.perf_counter()-t0,3),
 'teacher_targets':'synthetic shape placeholders; not an experiment'})
wrapped.zero_grad(set_to_none=True); wrapped.eval()

C:\Users\mateu\repos\tsfmxai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cpu', 'loss': 12.55541706085205, 'forecast_loss': 9.616202354431152, 'skeleton_shape': (12, 96), 'noise_shape': (12,), 'encoder_passes': 1, 'seconds': 0.758, 'teacher_targets': 'synthetic shape placeholders; not an experiment'}


Chronos2EquationModel(
  (chronos): Chronos2Model(
    (shared): Embedding(2, 768)
    (input_patch_embedding): ResidualBlock(
      (dropout): Dropout(p=0.1, inplace=False)
      (hidden_layer): Linear(in_features=48, out_features=3072, bias=True)
      (act): ReLU()
      (output_layer): Linear(in_features=3072, out_features=768, bias=True)
      (residual_layer): Linear(in_features=48, out_features=768, bias=True)
    )
    (patch): Patch()
    (instance_norm): InstanceNorm()
    (encoder): Chronos2Encoder(
      (block): ModuleList(
        (0-11): 12 x Chronos2EncoderBlock(
          (layer): ModuleList(
            (0): TimeSelfAttention(
              (self_attention): MHA(
                (q): Linear(in_features=768, out_features=768, bias=False)
                (k): Linear(in_features=768, out_features=768, bias=False)
                (v): Linear(in_features=768, out_features=768, bias=False)
                (o): Linear(in_features=768, out_features=768, bias=False)
          

## Batch e targets auxiliares

O adapter entrega rows por canal, `group_ids`, target futuro, targets tokenizados de equa??o e std em escala original. O professor congelado cria `teacher_embedding` na mesma ordem de equa??es usando o protocolo de relabel self-only e m?scara num?rica do notebook 03; targets para cross-channel n?o s?o suportados. a loss do skeleton n?o usa tokens/coefficient targets. Garantir que a ordem de linhas coincide com o flatten `[exemplo, canal]`.

## Sa?da esperada do wrapper (integra??o realizada)

O m?dulo reutiliz?vel `Chronos2EquationModel` captura os estados do encoder durante o forward Chronos normal (um passe, sem recomputar encode) e usa a m?dia dos patches de contexto como feature por canal. A chamada normal ao modelo calcula a loss quant?lica e as previs?es. A fun??o `equation_training_loss` soma a loss oficial com MSE do embedding teacher normalizado e Smooth-L1 para `log(std)` da inova??o.

A loss auxiliar foi exercitada com tensores sint?ticos e retropropaga??o: as heads receberam gradientes. Ainda ? necess?rio rodar a integra??o com checkpoint real; esta m?quina n?o tem CUDA e o pacote Chronos n?o est? instalado no ambiente do projeto. A configura??o recusa uma revis?o ausente para evitar um checkpoint flutuante.

Smoke executado com checkpoint Chronos-2 real em CPU: teacher placeholders de shape correto, loss finita e gradientes nas heads; nenhum optimizer step nem checkpoint salvo.

## Restri??es descobertas na integra??o

1. Chronos-2 `encode` inclui os embeddings dos patches futuros na sequ?ncia retornada. A head usa somente os primeiros `n_context_patches`, antes do bloco `[REG]`/patches futuros; verificar `n_context_patches` na vers?o pinada.
2. Chamamos `encode` e `forward` do modelo, duplicando atualmente o encoder por batch. Isso mant?m o caminho funcional inicial, mas ? ineficiente. Pr?xima otimiza??o: refatorar/decorar o forward do Chronos-2 para retornar hidden states no mesmo passe, sem alterar a loss oficial.
3. A instala??o local ? CPU-only e o checkout Chronos est? sob `experiments_local`; verificar vers?es/depend?ncias compat?veis antes do treino.
4. Teacher publicado ? single-series (vocab 37). Self-only equations may be relabeled locally; cross-channel equations require a six-series teacher (vocab 197).
5. O target de ru?do ? std das inova??es na unidade original. Para a head estimar isso, considerar fornecer loc/scale do Chronos ao caminho auxiliar; n?o assumir que os hidden states normalizados preservam escala.